In [1]:
!pip install -q -U peft transformers bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 91.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 646.8/646.8 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 100.2 MB/s eta 0:00:00


In [2]:
import json
import re
import pandas as pd
import torch
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

# --- 1. AUTHENTICATION ---
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

print("Loading Llama-3.1-8B-Instruct as Judge...")
model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

judge_model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    quantization_config=bnb_config, 
    device_map="auto"
)
judge_model.eval()
CHAT_DATA_PATH = "/kaggle/input/datasets/vasanthsubramanian01/chat-eval-dataset/chat_eval_dataset.json"
OUTPUT_CSV = "/kaggle/working/chatbot_metrics.csv"

def evaluate_chatbot_response(user_prompt, agent_response, scenario_type):
    eval_prompt = f"""
    You are an expert AI evaluator grading a Career Coach Chatbot.
    
    [SCENARIO TYPE]
    {scenario_type}
    
    [USER PROMPT]
    {user_prompt}
    
    [CHATBOT RESPONSE]
    {agent_response}
    
    Evaluate the response on a scale of 1 to 5 for the following metrics:
    1. Faithfulness (Hallucination): Does the bot rely on data/context rather than making up fake job titles, fake companies, or hallucinated resume details? (5 = Perfectly grounded, 1 = Highly hallucinated).
    2. Relevance: Does the response answer the user's specific prompt accurately and professionally? (5 = Perfect, 1 = Unhelpful/Rambling).
    
    Output ONLY valid JSON: {{"faithfulness": score, "relevance": score, "reasoning": "brief explanation"}}
    """
    
    messages = [
        {"role": "system", "content": "You output strict JSON only. No markdown formatting."},
        {"role": "user", "content": eval_prompt}
    ]
    
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(judge_model.device)
    
    with torch.no_grad():
        outputs = judge_model.generate(**inputs, max_new_tokens=150, do_sample=False)

    prompt_length = inputs.input_ids.shape[1]
    response_text = tokenizer.decode(outputs[0][prompt_length:], skip_special_tokens=True)
    
    try:
        # Extract JSON using regex to avoid Markdown artifacts
        match = re.search(r'\{.*\}', response_text.replace('\n', ''), re.DOTALL)
        if match:
            result = json.loads(match.group(0))
            return int(result.get("faithfulness", 0)), int(result.get("relevance", 0)), result.get("reasoning", "")
        return 0, 0, "No JSON found"
    except Exception as e:
        return 0, 0, f"Parse Error: {e}"

# [Cell 4] Execution Loop
with open(CHAT_DATA_PATH, "r") as f:
    eval_data = json.load(f)

results = []
for entry in tqdm(eval_data, desc="Grading Conversations"):
    faith_score, rel_score, reason = evaluate_chatbot_response(
        entry["user_prompt"],
        entry["agent_response"],
        entry["scenario_type"]
    )
    
    # Calculate Tool Accuracy: 1 if it behaved as expected, 0 if it hallucinated tool use or failed to use it
    tool_correct = 1 if entry["tool_used"] == entry["expected_tool"] else 0
    
    results.append({
        "resume_category": entry["resume_category"],
        "scenario_type": entry["scenario_type"],
        "tool_used": entry["tool_used"],
        "expected_tool": entry["expected_tool"],
        "tool_accuracy": tool_correct,
        "faithfulness_score": faith_score,
        "relevance_score": rel_score,
        "judge_reasoning": reason
    })

df = pd.DataFrame(results)
df.to_csv(OUTPUT_CSV, index=False)

print("\n--- EVALUATION SUMMARY ---")
print(f"Total Test Cases: {len(df)}")
print(f"Tool Utilization Accuracy: {(df['tool_accuracy'].mean() * 100):.2f}%")
print(f"Average Faithfulness (1-5): {df[df['faithfulness_score'] > 0]['faithfulness_score'].mean():.2f}")
print(f"Average Relevance (1-5): {df[df['relevance_score'] > 0]['relevance_score'].mean():.2f}")

# Group by Scenario
print("\n--- PERFORMANCE BY SCENARIO ---")
print(df.groupby("scenario_type")[["faithfulness_score", "relevance_score", "tool_accuracy"]].mean())

Loading Llama-3.1-8B-Instruct as Judge...


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Grading Conversations:   0%|          | 0/35 [00:00<?, ?it/s][transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Grading Conversations: 100%|██████████| 35/35 [04:16<00:00,  7.32s/it]


--- EVALUATION SUMMARY ---
Total Test Cases: 35
Tool Utilization Accuracy: 74.29%
Average Faithfulness (1-5): 4.40
Average Relevance (1-5): 4.91

--- PERFORMANCE BY SCENARIO ---
                           faithfulness_score  relevance_score  tool_accuracy
scenario_type                                                                
General_Advice_No_Context            4.714286         5.000000       0.428571
Match_Explanation                    4.142857         4.714286       0.571429
System_Doubt                         4.428571         4.857143       0.714286
Tool_Advice_No_Context               4.714286         5.000000       1.000000
Tool_Advice_With_Context             4.000000         5.000000       1.000000
